# Kategorija B: Colab Setup

Ovaj notebook sluzi da iz GitHub repoa podignes cijeli projekat u Google Colabu.

Za private repo ces mozda morati unijeti GitHub token ako Colab ne moze direktno da klonira repo.

Default evaluacija i inferenca ispod koriste ensemble `baseline_full_v1 + text_bgru_v1_cpu` plus learned face refiner, jer to trenutno daje najbolji validation skor i vidljivo jacu full-face mimiku u repou.

In [ ]:
GITHUB_USERNAME = "YOUR_GITHUB_USERNAME"
GITHUB_TOKEN = ""
REPO_NAME = "govorne-tehnologije-kategorija-b"
BRANCH = "master"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
CLONE_URL = REPO_URL if not GITHUB_TOKEN else f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
print(REPO_URL)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y git-lfs
!git lfs install
!git clone --branch {BRANCH} {CLONE_URL}
%cd /content/govorne-tehnologije-kategorija-b
!bash scripts/colab_bootstrap.sh --with-analysis

In [ ]:
!python scripts/evaluate.py --checkpoint artifacts/checkpoints/baseline_full_v1/best.pt artifacts/checkpoints/text_bgru_v1_cpu/best.pt --face-refiner artifacts/refiners/text_ensemble_face_refiner_v1.npz --device cuda --output-dir reports/figures/colab_ensemble_eval

In [ ]:
# Ako zelis novi trening u Colabu.
!python scripts/train.py --run-name improved_full_run --epochs 18 --batch-size 8 --device cuda --temporal-encoder bgru

In [ ]:
# Ako imas folder test_wavs sa WAV fajlovima.
# Ako imas i tekst, dodaj odgovarajuci folder test_texts sa .txt fajlovima istog naziva.
!python scripts/infer_folder.py --checkpoint artifacts/checkpoints/baseline_full_v1/best.pt artifacts/checkpoints/text_bgru_v1_cpu/best.pt --face-refiner artifacts/refiners/text_ensemble_face_refiner_v1.npz --input-dir test_wavs --text-dir test_texts --output-dir artifacts/predictions/colab_test --device cuda